In [33]:
# This cell can be deleted in the end.
%load_ext autoreload
%autoreload 2

import sys
sys.path.insert(1, '../')

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# 02 Interactive display
Sensors, impacts and virtual points can be added and moved in the 3D view interactively. The objects can then be moved around in the 3D view. The `pyFBS` supports also snapping of the objects to the surface of a predefined mesh (ussualy obtained from a STL file). When object snaps to the surface, not only the position of the object changes, but also the orientation of the object alligns with the normal of the mesh at the intersection (this feature can also be disabled). 

In [34]:
import pyFBS

import pandas as pd

## 3D View
Open 3Dviewer in the background.

In [35]:
view3D = pyFBS.view3D()

#### Add a structure
Load an example laboratory testbench and add a mesh to the 3D view.

In [36]:
stl = pyFBS.example_lab_testbench["STL"]["A"]
mesh = view3D.add_stl(stl,name = "ts",color = "#83afd2")

#### Datasets
Load a predefined datasets from an example. 

In [37]:
pos_xlsx = pyFBS.example_lab_testbench["meas"]["xlsx"]

df_sensors = pd.read_excel(pos_xlsx, sheet_name='Sensors_A')
df_impacts = pd.read_excel(pos_xlsx, sheet_name='Impacts_A')
df_vp = pd.read_excel(pos_xlsx, sheet_name='VP_Channels')

## Sensors
To enable interaction of sensors in the 3D view, just simply call a function `view3D.add_acc_dynamic(mesh,predefined = df_sensors)`. This will place the predefined sensors in the display and enable interaction with them and will allow you to add additional sensors. If you are starting a completely new analysis, you don't need the predefined data you can simply start with an empty dataset (i.e. `predefined = None`).

The object can be moved around by moving a black sphere in the 3D view. Arbitrary rotation around each local axis can be obtained my moving colored spheres (red - rotation around *X*, green - rotation around *Y*, blue - rotation around *Z*)

In [38]:
view3D.add_acc_dynamic(mesh,predefined = df_sensors)

The position and orientation data can be obtained by simply calling a function `view3D.get_acc_data()`:  

In [39]:
df_acc_updated = view3D.get_acc_data()
df_acc_updated

,Name,Description,Type,DirectionLabel,Quantity,Unit,Component,NodeNumber,Grouping,Position_1,Position_2,Position_3,Orientation_1,Orientation_2,Orientation_3
0,Sensor 1,None,None,None,None,None,None,None,None,-0.076519,0.142987,0.022,0,0,45.1658
1,Sensor 2,None,None,None,None,None,None,None,None,-0.0416791,0.278838,0.022,0,0,-50.2685


From the new positions and orientations of sensors a channel dataset can be generated (currently all the accelerometers are considered as tri-axial). If you have uni-axial accelerometers, redundant channels can be discarded afterwards.

In [40]:
df_chn_updated = pyFBS.utility.generate_channels_from_sensors(df_acc_updated)
df_chn_updated

,Name,Description,Type,DirectionLabel,Quantity,Unit,Component,NodeNumber,Grouping,Position_1,Position_2,Position_3,Direction_1,Direction_2,Direction_3
0,Sensor 1x,None,None,None,None,None,None,None,None,-0.076519,0.142987,0.022,0.705057,0.70915,0
1,Sensor 1y,None,None,None,None,None,None,None,None,-0.076519,0.142987,0.022,-0.70915,0.705057,0
2,Sensor 1z,None,None,None,None,None,None,None,None,-0.076519,0.142987,0.022,0,0,1
3,Sensor 2x,None,None,None,None,None,None,None,None,-0.0416791,0.278838,0.022,0.639191,-0.769048,-0
4,Sensor 2y,None,None,None,None,None,None,None,None,-0.0416791,0.278838,0.022,0.769048,0.639191,0
5,Sensor 2z,None,None,None,None,None,None,None,None,-0.0416791,0.278838,0.022,0,-0,1


If you have the channel dataset (tri-axial) you can generate the sensor dataset. This transformation is not unique and gimbal lock problem can arise. In this case third angle (rotation around Z axis) is set to zero and a warning is raised (see [scipy.spatial.transform.Rotation.as_euler](https://docs.scipy.org/doc/scipy/reference/generated/scipy.spatial.transform.Rotation.as_euler.html) for more details). Nevertheless, the obtained rotation angles still represent the correct rotation.

In [31]:
df_acc_from_chn = pyFBS.utility.generate_sensors_from_channels(df_chn_updated)
df_acc_from_chn

,Name,Description,Type,DirectionLabel,Quantity,Unit,Component,NodeNumber,Grouping,Position_1,Position_2,Position_3,Orientation_1,Orientation_2,Orientation_3
0,S1,None,None,None,None,None,None,None,None,-0.076519,0.142987,0.022,0,0,45.1658
1,S2,None,None,None,None,None,None,None,None,-0.0416791,0.278838,0.022,0,0,-50.2685
2,S3,None,None,None,None,None,None,None,None,-0.156679,0.211667,-0.008,-90,-1.2668e-09,-1.35736e-10
3,S4,None,None,None,None,None,None,None,None,0.0553074,0.359847,0.00810314,-3.12308e-09,2.10453e-08,40


## Impacts

Adding impacts interactively to the 3D view is the same, only the object display is different.

In [32]:
view3D.add_imp_dynamic(mesh,predefined = df_impacts)

The updated positions and orientation of the virtual points can be obtained directly:

In [13]:
df_imp_updated = view3D.get_imp_data()
df_imp_updated

,Name,Description,Type,DirectionLabel,Quantity,Unit,Component,NodeNumber,Grouping,Position_1,Position_2,Position_3,Direction_1,Direction_2,Direction_3
0,Impact 1,None,None,None,None,None,None,None,None,0.0143109,0.207959,0.017,1.43845e-10,0,-1
1,Impact 2,None,None,None,None,None,None,None,None,-0.0189236,0.31811,0.00729185,0.642787,-0.766045,0
2,Impact 3,None,None,None,None,None,None,None,None,-0.112749,0.157749,0.00735692,0.707107,0.707107,2.18472e-10
3,Impact 4,None,None,None,None,None,None,None,None,-0.0839125,0.243174,0.017,1.29315e-10,0,-1
4,Impact 5,None,None,None,None,None,None,None,None,0.03,0.153247,0.00723164,-1,0,0
5,Impact 6,None,None,None,None,None,None,None,None,0.0151714,0.048171,0.017,0,1.6434e-10,-1
6,Impact 7,None,None,None,None,None,None,None,None,0.0143109,0.207959,0.017,1.17886e-10,0,-1
7,Impact 8,None,None,None,None,None,None,None,None,-0.0189236,0.31811,0.00729185,0.642787,-0.766045,0
8,Impact 9,None,None,None,None,None,None,None,None,-0.112749,0.157749,0.00735692,0.707107,0.707107,1.88098e-10
9,Impact 10,None,None,None,None,None,None,None,None,-0.0839125,0.243174,0.017,0,0,-1


## Virtual points

Adding virtual points interactively to the 3D view is the same, only the object display is different.

In [14]:
view3D.add_vp_dynamic(mesh,predefined = df_vp)

The updated positions and orientation of the virtual points can be obtained directly:

In [15]:
df_vp_updated = view3D.get_vp_data()
df_vp_updated

,Name,Description,Type,DirectionLabel,Quantity,Unit,Component,NodeNumber,Grouping,Position_1,Position_2,Position_3,Orientation_1,Orientation_2,Orientation_3
0,VP 1,None,None,None,None,None,None,None,None,0.0340844,0.348107,0.007,0,0,0


## Output
You can save the new positions and orientations of the objects in Excel file in a simple manner:  

In [ ]:
with pd.ExcelWriter('./output_file.xlsx') as writer:  
    df_acc_updated.to_excel(writer, sheet_name='Sensors',index = False)
    df_imp_updated.to_excel(writer, sheet_name='Impacts',index = False)
    df_chn_updated.to_excel(writer, sheet_name='Channels',index = False)